# Intro

In this Jupyter Notebook I try to break-down the collect_mutations.py script to understand each step and update it so that it will be compatible with my own version of the upstream analysis.

In [6]:
# loading packages

import re
import os
import sys
from tqdm import tqdm
import pickle as pkl
import pandas as pd
from pandas.api.types import is_string_dtype
from pandas.api.types import is_numeric_dtype


In [2]:

print(os.getcwd()) 


/Users/alimos313/Documents/studies/phd/research/genome-evo-proj/code/snakemake-scripts


### Break-down of the first function (read_files)

This function iterates over mutation output files of each sample and collect them in one file.

In [22]:
# params

filename1 = "/Users/alimos313/Documents/studies/phd/research/genome-evo-proj/data/processed-data/mutations/iii/13/13MT2EXPIIIVP190seq10102019_S1_L001.csv"
filename2 = "/Users/alimos313/Documents/studies/phd/research/genome-evo-proj/data/processed-data/mutations/iii/13/13MT2EXPIIIVP230seq10102019_S5_L001.csv"
my_filename1 = "/Users/alimos313/Desktop/scrap/13MT2EXPIIIVP190seq10102019_S1_L001.csv"
my_filename2 = "/Users/alimos313/Desktop/scrap/13MT2EXPIIIVP230seq10102019_S5_L001.csv"
lines=None
passage=None

In [23]:

deletions = []
insertions = []
mutations = []

counter = 0
# read per line and split by comma to get a list of relevant filed for each mutation

with open(filename1) as f:
    for line in f.readlines():
        if line.strip().split(',')[0] == "M":
            if counter < 2:
                counter+=1
                fields = line.strip().split(',')
                
                if lines is None:
                    lines = fields[6]
                if passage is None:
                    passage = int(fields[7])
                try:
                    from_pos = int(fields[1])
                except ValueError:
                    continue
                except IndexError:
                    print(line, lines,passage, 1)
                
                if fields[0] == 'D': #deletions: start, stop, fraction, reads, coverage at pos
                    end_pos = int(fields[2])
                    deletions.append([from_pos, end_pos, float(fields[3]),int(fields[4]),int(fields[5]),
                                        lines,passage])
                elif fields[0] == 'I': #insertions: position, insterted bases, fraction, reads, coverage at pos
                    inserted = fields[2]
                    insertions.append([from_pos, inserted, float(fields[3]),int(fields[4]),int(fields[5]),
                                        lines,passage])
                elif fields[0] == 'M':
                    new_base = fields[2]
                    mutations.append([from_pos,new_base,float(fields[3]),int(fields[4]),int(fields[5]),lines,
                                    passage])



print(fields[:12])
print(lines, passage)
print(from_pos)
print(mutations)

['M', '536', 'A', '0.0011196118678857996', '6', '5359', 'III', '190']
III 190
536
[[485, 'A', 0.002296211251435132, 4, 1742, 'III', 190], [536, 'A', 0.0011196118678857996, 6, 5359, 'III', 190]]


In [55]:

def flatten(l):
    return [item for sublist in l for item in sublist]

def read_files(filename,lines=None,passage=None):
    deletions = []
    insertions = []
    mutations = []

    with open(filename) as f:
        for line in f.readlines():
            fields = line.strip().split(',')
            #mutid = (fields[0],fields[1],fields[2])
            if lines is None:
                lines = fields[8].strip()
            if passage is None:
                passage = int(fields[9])
            try:
                from_pos = int(fields[1])
                end_pos = int(fields[2])
                ref_base = str(fields[3])
                alt_base = str(fields[4])
                
            except ValueError:
                continue
            except IndexError:
                print(line, lines,passage, 1)
            if fields[0] == 'D' or fields[0] == 'LD': #deletions: start, stop, fraction, reads, coverage at pos
                
                deletions.append([from_pos, end_pos, ref_base, alt_base, fields[0], float(fields[5]),int(fields[6]),int(fields[7]),
                                  lines,passage])
            elif fields[0] == 'I': #insertions: position, insterted bases, fraction, reads, coverage at pos
                
                insertions.append([from_pos, end_pos, ref_base, alt_base, fields[0], float(fields[5]),int(fields[6]),int(fields[7]),
                                   lines,passage])
            elif fields[0] == 'M':
                
                mutations.append([from_pos, end_pos, ref_base, alt_base, fields[0], float(fields[5]),int(fields[6]),int(fields[7]),
                                  lines,passage])
    try:
        if mutations[-1][0] < 9000:
            print(filename, mutations[-1][0], 2)
    except IndexError:
        print(lines,filename[filename.find('VP')+2:filename.find('.')],3)
    return deletions,insertions,mutations


variants = []

files = [my_filename1, my_filename2]
print(files)

for line in files:
#    line_name = my_line_translation[line.split('/')[2]]
    line_nu = line.split('/')[-1][:2]
    passage = int(re.findall(r'VP([0-9]+)',line)[-1])
    print(passage,line_nu)
    d,i,m = read_files(line,line_nu,passage)
    
    variants += d,i,m
    

variants = pd.DataFrame(flatten(variants),columns=['start','end','ref','alt','mut_type','fraction','reads','coverage','line','passage'])


variants.sort_values(by='fraction',ascending=False,inplace=True)



# mask = [item == "I" for item in variants.loc[:,"mut_type"].tolist()]
# print(variants[mask].head())


variants.to_pickle('/Users/alimos313/Desktop/scrap/all_variants.pkl',protocol=2)



# convert the pkl object to csv for further analyses
with open("/Users/alimos313/Desktop/scrap/all_variants.pkl", "rb") as f:
    object = pkl.load(f)
    
df = pd.DataFrame(object)
df.to_csv("/Users/alimos313/Desktop/scrap/all_variants.csv", index=False)


['/Users/alimos313/Desktop/scrap/13MT2EXPIIIVP190seq10102019_S1_L001.csv', '/Users/alimos313/Desktop/scrap/13MT2EXPIIIVP230seq10102019_S5_L001.csv']
190 13
230 13
